# 05 — Gerar dados mock (SQLite)

Popula `/content/drive/MyDrive/AssistenteHospitalar/files/hospital.db` com 50 pacientes sintéticas cobrindo cenários do enunciado:
- gestantes
- climatério
- mamografia / papanicolau em atraso
- vítimas de violência (com notificação SINAN)
- usuárias de método contraceptivo

## Setup — onde a pasta `lib/` precisa estar

Esse notebook importa `from lib import db, mock_data`. Deixe a pasta `lib/` em **uma** das duas localizações:

**Opção A — upload no Drive (persiste entre sessões):**
```
/MyDrive/AssistenteHospitalar/lib/
```
Use a configuração default abaixo (`LIB_PATH = DRIVE_BASE`).

**Opção B — git clone (recomendado se repo está no GitHub):**
```python
!git clone https://github.com/<seu-user>/<repo>.git /content/repo
LIB_PATH = '/content/repo'
```

**Opção C — drag-and-drop direto no Colab (rápido, perde ao fechar sessão):**
Arraste a pasta `lib/` do repo local pra árvore `/content/` do Colab e use `LIB_PATH = '/content'`.

In [ ]:
!pip install -q faker

In [ ]:
import os
import sys
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive', force_remount=True)

DRIVE_BASE = '/content/drive/MyDrive/AssistenteHospitalar'
LIB_PATH   = DRIVE_BASE           # lib/ deve estar em /MyDrive/AssistenteHospitalar/lib/
DB_PATH    = f'{DRIVE_BASE}/files/hospital.db'

sys.path.insert(0, LIB_PATH)
os.environ['HOSPITAL_DB_PATH'] = DB_PATH

from lib import db, mock_data
print('DB path :', DB_PATH)
print('Lib path:', LIB_PATH)

In [ ]:
conn = db.connect()
db.reset_database(conn)   # apaga e recria schema — não usar em produção
sumario = mock_data.populate(conn, n_pacientes=50, seed=42)
print('\nBanco populado.')

In [ ]:
# Verificação: contagens por tabela
for tabela in ['pacientes','prontuario_gineco','exames','ciclos_menstruais',
               'registros_violencia','medicamentos','log_acesso']:
    n = conn.execute(f'SELECT COUNT(*) AS c FROM {tabela}').fetchone()['c']
    print(f'  {tabela:<22} {n}')

In [ ]:
# Amostra: paciente 1 (full picture)
import json
p = conn.execute('SELECT * FROM pacientes WHERE paciente_id = 1').fetchone()
pron = conn.execute('SELECT * FROM prontuario_gineco WHERE paciente_id = 1').fetchone()
exames = conn.execute('SELECT tipo, data_realizacao, resultado FROM exames WHERE paciente_id = 1').fetchall()

print('Paciente:', dict(p))
print('Prontuário:', dict(pron) if pron else None)
print('Exames:')
for e in exames:
    print('  -', dict(e))

In [ ]:
# Sanity checks dos cenários
print('Pacientes ≥50a com mamografia (qualquer):')
rows = conn.execute('''
    SELECT p.paciente_id, p.nome,
           (julianday('2026-05-22') - julianday(p.data_nascimento))/365.0 AS idade,
           MAX(e.data_realizacao) AS ultima_mamo
    FROM pacientes p
    LEFT JOIN exames e ON e.paciente_id = p.paciente_id AND e.tipo='mamografia'
    WHERE (julianday('2026-05-22') - julianday(p.data_nascimento))/365.0 BETWEEN 50 AND 69
    GROUP BY p.paciente_id
    ORDER BY ultima_mamo IS NULL DESC, ultima_mamo ASC
    LIMIT 10
''').fetchall()
for r in rows:
    print(f'  id={r["paciente_id"]:>3}  idade={int(r["idade"])}  última mamo: {r["ultima_mamo"]}')

print('\nRegistros de violência (resumo por tipo):')
rows = conn.execute('SELECT tipo, COUNT(*) AS n FROM registros_violencia GROUP BY tipo').fetchall()
for r in rows:
    print(f'  {r["tipo"]:<14} {r["n"]}')

In [ ]:
conn.close()
print('Banco salvo em:', DB_PATH)